In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

matches = pd.read_csv(f'{BASE}/matches.csv')
matches['Date'] = pd.to_datetime(matches['match_id'].str.split('-').str[0], format='%Y%m%d', errors='coerce')

print('matches: ', matches.shape)

matches:  (7569, 64)


In [3]:
# shuffle: trocar p1/p2 aleatoriamente para remover viés de quem serve primeiro
np.random.seed(42)
swap_mask = np.random.rand(len(matches)) < 0.5

p1_cols = [c for c in matches.columns if c.startswith('p1_')]
p2_cols = [c.replace('p1_', 'p2_') for c in p1_cols]

for p1_col, p2_col in zip(p1_cols, p2_cols):
    matches.loc[swap_mask, [p1_col, p2_col]] = matches.loc[swap_mask, [p2_col, p1_col]].values

matches.loc[swap_mask, 'Player 1'], matches.loc[swap_mask, 'Player 2'] = (
    matches.loc[swap_mask, 'Player 2'].values,
    matches.loc[swap_mask, 'Player 1'].values
)

matches.loc[swap_mask, 'Pl 1 hand'], matches.loc[swap_mask, 'Pl 2 hand'] = (
    matches.loc[swap_mask, 'Pl 2 hand'].values,
    matches.loc[swap_mask, 'Pl 1 hand'].values
)

matches.loc[swap_mask, 'Winner'] = matches.loc[swap_mask, 'Winner'].map({1.0: 2.0, 2.0: 1.0})

In [4]:
# calcular diffs
p1_cols = [c for c in matches.columns if c.startswith('p1_')]
p2_cols = [c.replace('p1_', 'p2_') for c in p1_cols]

for p1_col, p2_col in zip(p1_cols, p2_cols):
    if p2_col in matches.columns and matches[p1_col].dtype in ['float64', 'int64']:
        matches[p1_col.replace('p1_', 'diff_')] = matches[p1_col] - matches[p2_col]

# dropar colunas sem valor preditivo
drop_cols = ['match_id', 'Player 1', 'Player 2', 'Time', 'Court', 'Umpire',
             'Final TB?', 'Charted by', 'Date', 'p1_Surface', 'p2_Surface',
             'p1_hand', 'p2_hand']
matches = matches.drop(columns=[c for c in drop_cols if c in matches.columns])

# dropar p1_ e p2_ absolutas, manter só diffs + rating + categoricas
abs_cols = [c for c in matches.columns if c.startswith('p1_') or c.startswith('p2_')]
matches = matches.drop(columns=abs_cols)

matches.shape

(7569, 29)

In [5]:
# label encoding nas categoricas
cat_cols = matches.select_dtypes(include='object').columns.tolist()

encoders = {}
for col in cat_cols:
    encoders[col] = LabelEncoder()
    matches[col] = encoders[col].fit_transform(matches[col].astype(str))

# target e split
matches_clean = matches.dropna(subset=['Winner'])
matches_clean['p1_wins'] = (matches_clean['Winner'] == 1.0).astype(int)

X = matches_clean.drop(columns=['Winner', 'p1_wins'])
y = matches_clean['p1_wins']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)

X_train: (6028, 28)
X_test:  (1508, 28)


C:\Users\lucas\AppData\Local\Temp\ipykernel_61888\4038025681.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_clean['p1_wins'] = (matches_clean['Winner'] == 1.0).astype(int)


In [6]:
X_train.isnull().sum().sort_values()

Pl 1 hand                           0
Pl 2 hand                           0
Tournament                          0
Round                               0
Surface                             0
Best of                             0
diff_rating_before                  3
diff_rating_surface_before          3
diff_avg_second_serve_won_pct     693
diff_avg_bpo_ue_pct               693
diff_avg_bpo_conv_pct             693
diff_avg_first_serve_won_pct      693
diff_avg_bp_clutch_save_pct       693
diff_avg_winners_fh_ratio         693
diff_avg_bp_first_in_pct          693
diff_avg_winners_per_pt           693
diff_avg_return_won_pct           693
diff_avg_df_pct                   693
diff_avg_ace_pct                  693
diff_avg_first_serve_pct          693
diff_avg_ue_per_pt                693
diff_avg_bp_save_pct              705
diff_avg_srv_win_1_3             1107
diff_avg_srv_win_4_6             1107
diff_avg_srv_win_7plus           1107
diff_avg_ret_win_1_3             1155
diff_avg_ret

In [7]:
X_train.columns

Index(['Pl 1 hand', 'Pl 2 hand', 'Tournament', 'Round', 'Surface', 'Best of',
       'diff_avg_first_serve_pct', 'diff_avg_first_serve_won_pct',
       'diff_avg_second_serve_won_pct', 'diff_avg_ace_pct', 'diff_avg_df_pct',
       'diff_avg_return_won_pct', 'diff_avg_winners_per_pt',
       'diff_avg_ue_per_pt', 'diff_avg_winners_fh_ratio',
       'diff_avg_bp_save_pct', 'diff_avg_bp_clutch_save_pct',
       'diff_avg_bp_first_in_pct', 'diff_avg_bpo_conv_pct',
       'diff_avg_bpo_ue_pct', 'diff_avg_srv_win_1_3', 'diff_avg_srv_win_4_6',
       'diff_avg_srv_win_7plus', 'diff_avg_ret_win_1_3',
       'diff_avg_ret_win_4_6', 'diff_avg_ret_win_7plus', 'diff_rating_before',
       'diff_rating_surface_before'],
      dtype='object')

Criando o pipeline

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols),
], remainder='passthrough')

#### Testando modelos

Primeiramente, vamos fazer um modelo de benchmark para comparar com o nosso. Se for empatado, vamos dar vitória a um aleatório. Quem tiver o elo maior, vamos dar vitória

**Benchmark**

In [16]:
y_bench = X_train.apply(
    lambda row: 1 if row['diff_rating_before'] > 0 else np.random.randint(0,1),
    axis=1
)

In [17]:
from sklearn.metrics import f1_score

f1_score(y_train, y_bench)

0.6651308211288465

**XGBoost**

In [10]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [19]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
    )),
])

scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='f1')
y_pred = cross_val_predict(pipeline, X_train, y_train, cv=3)
print(f'f1: {scores.mean():.4f} ± {scores.std():.4f}')


f1: 0.6613 ± 0.0079


**Redes Neurais**

In [21]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline_mlp = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),  # MLP é sensível à escala
    ('model', MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )),
])

scores = cross_val_score(pipeline_mlp, X_train, y_train, cv=5, scoring='f1')
print(f'f1: {scores.mean():.4f} ± {scores.std():.4f}')


f1: 0.6641 ± 0.0058


**Logistic Regression**

In [26]:
from sklearn.linear_model import LogisticRegression

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
     
])

scores = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='f1')
print(f'f1: {scores.mean():4f} + {scores.std():.4f}')

f1: 0.668917 + 0.0098
